# Counting Communities (Group 20)

## 1. Introduction

...

In [7]:
# load libraries
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os, requests, zipfile, gzip, io
from scipy.stats import poisson
from types import SimpleNamespace
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

SEED = 2026

DATASETS = "data"
os.makedirs(DATASETS, exist_ok=True)

WEAVER_URL = "https://nrvis.com/download/data/dynamic/aves-weaver-social.zip"

def download(url, file_name):
    path = os.path.join(DATASETS, file_name)
    if not os.path.exists(path):
        print(f"downloading {file_name} ...")
        try:
            req = requests.get(url)
            req.raise_for_status() # in case of errors

            with open(path, 'wb') as f:
                f.write(req.content)

        except Exception as error:
            if os.path.exists(path):
                os.remove(path) # clean corrupted install
            raise RuntimeError(f"Graph download failed : {error}. If it continues to fail, download them by hand and save them in a 'data' folder in the root directory")

    return path

def csv_path(name):
    return os.path.join(DATASETS, name)

def save_edges(G, file_name):
    nx.to_pandas_edgelist(G)[["source", "target"]].to_csv(csv_path(file_name), index=False)

def read_edges(file_name, nodes=None):
    edges = pd.read_csv(csv_path(file_name), keep_default_na=False)
    G = nx.Graph()
    if nodes is not None:
        G.add_nodes_from(nodes)
    G.add_edges_from(zip(edges["source"], edges["target"]))
    return G

def load_weaver():
    # 3-column edge list: source, target, timestamp -> collapse to static undirected graph
    cached = "aves_weaver_social_edges.csv"
    if not os.path.exists(csv_path(cached)):
        zipped_folder = zipfile.ZipFile(download(WEAVER_URL, "aves-weaver-social.zip"))
        edges = pd.read_csv(
            io.BytesIO(zipped_folder.read("aves-weaver-social.edges")),
            header=None,
            sep=r"\s+", engine='python', comment="#", 
            names=["source", "target", "timestamp"]
        )
        # same pair may interact many times; keep each unique undirected edge once
        lo = edges[["source", "target"]].min(axis=1)
        hi = edges[["source", "target"]].max(axis=1)
        pairs = pd.DataFrame({"source": lo, "target": hi}) \
            .query("source != target") \
            .drop_duplicates()
        pairs.to_csv(csv_path(cached), index=False)
    return read_edges(cached)

weaver = load_weaver()
print("Weaver : ", weaver)

downloading aves-weaver-social.zip ...
Weaver :  Graph with 445 nodes and 1332 edges


## 2. Community Count Estimator

...

### 2.1 Community Detection Algorithm

...

### 2.2 Network Properties Analysis

...

In [14]:
def describe_network(G, name):
    degree_list = np.array([deg for _, deg in G.degree()])
    print(f"Computing network {name}")
    return {"network": name, "n": G.number_of_nodes(), "m": G.number_of_edges(),
            "mean degree": degree_list.mean(), "max degree": int(degree_list.max()),
            "leaves": int((degree_list == 1).sum()),
            "isolated vertices": int((degree_list == 0).sum()),
            "interactions": nx.number_connected_components(G),
            "avg clustering coeff.": nx.average_clustering(G),
            "assortativity": nx.degree_assortativity_coefficient(G),
            "transitivity": nx.transitivity(G)}

display(pd.DataFrame([describe_network(weaver, "Social Weaver Network")]).set_index("network"))

Computing network Social Weaver Network


,n,m,mean degree,max degree,leaves,isolated vertices,interactions,avg clustering coeff.,assortativity,transitivity
network,,,,,,,,,,
Social Weaver Network,445,1332,5.986517,27,41,0,22,0.668505,0.199066,0.588094


### 2.3 Predictions Based on Network Properties

-> global clustering coefficient